---
jupyter: python3
author: Corey T. White
execute:
  eval: true
  freeze: auto
date-modified: today
format:
  html:
    toc: true
    code-tools: true
    code-copy: true
    code-fold: false
---

# SSURGO Soil Data in GRASS

In [ ]:
# | label: imports
# | echo: false
import os
import subprocess
import sys

# import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
from PIL import Image
import pandas as pd
from IPython.display import IFrame
import numpy as np

# import seaborn as sns
import pandas as pd

# Ask GRASS GIS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

# Import the GRASS GIS packages we need.
import grass.script as gs

# Import GRASS Jupyter
import grass.jupyter as gj
from grass.tools import Tools

In [ ]:
# | label: init_session
gisdb = os.path.join(os.getenv("HOME"), "grassdata")
mapset = "ssurgo_demo"
session = gj.init(gisdb, "nc_spm_full_v2alpha2", "PERMANENT")
tools = Tools(session=session)
try:
    tools.g_mapset(mapset=mapset, flags="c")
except Exception as e:
    print(f"{e}")

Install the `r.in.ssurgo` extension if it is not already installed. This extension is available in the GRASS Addons repository and can be installed using `g.extension`. The source code for this extension is also available in the `tools/r.soildb/r.in.ssurgo` directory of this repository.

In [ ]:
# | label: install_extension
# | eval: false
tools.g_extension(extension="r.in.ssurgo", url="../r.in.ssurgo")

Set the computational region to the elevation raster and create a relief raster for hillshading.

In [ ]:
# | label: set_region
region_text = tools.g_region(raster="elevation", flags="p").text
tools.r_relief(input="elevation", output="relief")
print(region_text)

Import SSURGO data using `r.in.ssurgo`. Note that this will import the soil areas map, hydrologic group map, and ksat maps for low, regular, and high values. The `mukey` column is also imported as a raster with a categorical color scheme applied. The hydrologic group map also has a categorical color scheme applied.

In [ ]:
# | label: import_ssurgo
# | eval: false
tools.r_in_ssurgo(
    # ssurgo_path="../data/gSSURGO_CONUS.zip/gSSURGO_CONUS.gdb",
    soils="soil_areas",
    hydgrp="hydgrp",
    ksat_l="ksat_l",
    ksat_r="ksat_r",
    ksat_h="ksat_h",
    mukey="mukey",
)

In [ ]:
# | label: raster_maps
print(tools.g_list(type="raster", mapset=mapset).text)

In [ ]:
print(tools.g_list(type="vector", mapset=mapset).text)

In [ ]:
print(tools.v_info(map="soil_areas").text)

In [ ]:
print(tools.v_info(map="soil_areas", flags="c").text)

In [ ]:
# | label: soil attributes
json_data = tools.v_db_select(map="soil_areas", format="json").json
df = pd.DataFrame(json_data["records"])
df.head()

In [ ]:
# | label: list_rasters
print(tools.g_list(type="raster", mapset=mapset).text)

## MUKEY Map

In [ ]:
# | label: mukey
m = gj.Map(use_region=True, filename="mukey.png")
m.d_shade(shade="relief", color="mukey")
# m.d_legend(raster="mukey", title="MUKEY", flags="b")
m.show()

## Ksat Maps

In [ ]:
# | label: ksat
# | layout-ncol: 3
# | fig-cap: "Ksat maps for low, regular, and high values"
# | fig-subcap:
# |     - Low
# |     - Regular
# |     - High

with gs.RegionManager(w="w-4000"):
    for raster in ["ksat_l", "ksat_r", "ksat_h"]:
        m = gj.Map(use_region=True, filename=f"{raster}.png")
        m.d_shade(shade="relief", color=raster)
        m.d_legend(
            raster=raster,
            title=f"{raster} (mm/hr)",
            digits=0,
            flags="bst",
            at=[25, 85, 4, 13],
            border_color="none",
        )
        m.d_barscale(
            flags="n",
            at=[0, 10],
            color="black",
            style="both_ticks",
            length=2,
            units="kilometers",
            width_scale=1,
            bgcolor="none",
            segment=4,
            fontsize=10,
        )
        m.show()

## Hydrologic Group

In [ ]:
# | label: hydrologic_group_map
with gs.RegionManager(w="w-10700"):
    m = gj.Map(use_region=True, filename="hydgrp.png")
    m.d_shade(shade="relief", color="hydgrp")
    m.d_legend(
        raster="hydgrp",
        title="Hydrologic Group",
        flags="bcn",
        at=[25, 85, 2, 18],
        border_color="none",
    )
    m.d_barscale(
        flags="n",
        at=[2, 18],
        color="black",
        style="both_ticks",
        length=5,
        units="kilometers",
        width_scale=1,
    )
    m.show()

## Soil Texture (Sand / Silt / Clay)

The import populates `soil_areas` with depth-weighted texture columns
(`sandtotal_r`, `silttotal_r`, `claytotal_r`) over the horizon window
`[hzdept_r, hzdepb_r]`. Rasterize them with `v.to.rast`.


In [ ]:
# | label: soil_texture_rasters
for col, out in [
    ("sandtotal_r", "sand_r"),
    ("silttotal_r", "silt_r"),
    ("claytotal_r", "clay_r"),
]:
    tools.v_to_rast(
        input="soil_areas",
        type="area",
        use="attr",
        attribute_column=col,
        output=out,
        overwrite=True,
    )
    tools.r_colors(map=out, color="ryg", flags="e")

In [ ]:
# | label: soil_texture_maps
# | layout-ncol: 3
# | fig-cap: "Depth-weighted soil texture (% by mass)"
# | fig-subcap:
# |     - Sand
# |     - Silt
# |     - Clay

for raster, title in [("sand_r", "Sand %"), ("silt_r", "Silt %"), ("clay_r", "Clay %")]:
    with gs.RegionManager(w="w-4000"):
        m = gj.Map(use_region=True)
        m.d_rast(map=raster)
        m.d_shade(shade="relief", color=raster)
        m.d_legend(
            raster=raster,
            title=title,
            flags="bst",
            at=[25, 85, 4, 14],
            border_color="none",
        )
        m.d_barscale(
            flags="n",
            at=[1, 15],
            color="black",
            style="both_ticks",
            length=2,
            units="kilometers",
            width_scale=1,
            bgcolor="none",
            segment=4,
        )
        m.show()

## Available Water Capacity & Organic Matter

`awc_r` (cm/cm) and `om_r` (% by mass) are also depth-weighted means over the
horizon window.


In [ ]:
# | label: awc_om_rasters
for col, out in [("awc_r", "awc"), ("om_r", "om")]:
    tools.v_to_rast(
        input="soil_areas",
        type="area",
        use="attr",
        attribute_column=col,
        output=out,
        overwrite=True,
    )
    tools.r_colors(map=out, color="bcyr")

In [ ]:
# | label: awc_om_maps
# | layout-ncol: 2
# | fig-cap: "Plant-available water capacity and organic matter"
# | fig-subcap:
# |     - "AWC (cm/cm)"
# |     - "Organic matter (%)"

for raster, title in [("awc", "AWC (cm/cm)"), ("om", "OM (%)")]:
    with gs.RegionManager(w="w-4000"):
        m = gj.Map(use_region=True, filename=f"{raster}.png")
        m.d_rast(map=raster)
        m.d_shade(shade="relief", color=raster)
        m.d_legend(
            raster=raster,
            title=title,
            flags="bst",
            at=[28, 88, 3, 13],
            border_color="none",
        )
        m.d_barscale(
            flags="n",
            at=[1, 15],
            color="black",
            style="both_ticks",
            length=2,
            units="kilometers",
            width_scale=1,
            bgcolor="none",
            segment=4,
        )
        m.show()

## 3D Depth-Sliced Ksat (r3 workflow)

Setting `depths=` switches the Ksat outputs from 2D depth-weighted rasters to
**3D rasters** with one slice per depth bin. Internally the addon writes
per-slice attribute columns (`ksat_r__s0`, `ksat_r__s1`, ...) on the soils
vector, rasterizes each, then stacks them via `r.to.rast3`.

Boundaries are in centimetres. The example below produces four slices:
`[0–10), [10–30), [30–60), [60–100)` cm.


In [ ]:
# | label: import_ssurgo_3d
# | eval: false
tools.r_in_ssurgo(
    soils="soil_areas_3d",
    ksat_l="ksat_l_3d",
    ksat_r="ksat_r_3d",
    ksat_h="ksat_h_3d",
    depths="0,10,30,60,100",
    overwrite=True,
)

In [ ]:
# | label: raster3d_maps
print(tools.g_list(type="raster_3d", mapset=mapset).text)

In [ ]:
# | label: r3_info
print(tools.r3_info(map="ksat_r_3d").text)

In [ ]:
# | label: r3_slice_extract
# Expand the 3D ksat_r raster into one 2D map per depth slice.
tools.r3_to_rast(input="ksat_r_3d", output="ksat_r_slice", flags="r", overwrite=True)
slice_rasters = sorted(
    tools.g_list(type="raster", pattern="ksat_r_slice*", format="json").json,
    key=lambda x: x["name"],
)

print("\n".join([r["name"] for r in slice_rasters]))

In [ ]:
# | label: r3_slice_maps
# | layout-ncol: 2
# | fig-cap: "Ksat (regular) by depth slice"
# | fig-subcap:
# |     - "0–10 cm"
# |     - "10–30 cm"
# |     - "30–60 cm"
# |     - "60–100 cm"

titles = [
    "0-10 cm",
    "10-30 cm",
    "30-60 cm",
    "60-100 cm",
]
mapnames_and_title = [(r["name"], title) for r, title in zip(slice_rasters, titles)]
for raster, title in mapnames_and_title:
    with gs.RegionManager(w="w-4000"):
        m = gj.Map(use_region=True)
        m.d_rast(map=raster)
        m.d_shade(shade="relief", color=raster)
        m.d_legend(
            raster=raster,
            title=title,
            digits=0,
            flags="bst",
            at=[25, 85, 4, 13],
            border_color="none",
        )
        m.d_barscale(
            flags="n",
            at=[0, 10],
            color="black",
            style="both_ticks",
            length=2,
            units="kilometers",
            width_scale=1,
            bgcolor="none",
            segment=4,
            fontsize=10,
        )
        m.show()

In [ ]:
# | label: nviz_ksat_raster3d_map
# | eval: false
# | fig-cap: "3D isosurfaces of Ksat (regular) over the 0-100 cm soil column."

# Surprise from m.nviz.image: the first token of `isosurf_level=` is a
# 1-based INDEX into the volume= list, NOT the volume name (despite what the
# manual says). With volume="ksat_r_3d", the index for that volume is 1, so
# levels are "1:<value>,1:<value>,...". Using the name produces atoi() == 0
# and the failure "Unable to add isosurface (volume set 0)".
#
# zexag is the main visibility knob: the volume's z-range is 0-100 (cm) but
# xy is in metres, so without strong z exaggeration the volume collapses to
# a thin sheet. Explicit colors per shell (low Ksat -> brown, mid -> tan,
# high -> blue) track the 2D ksat palette; transparency descends so all
# four shells are visible at once.
with gs.RegionManager(raster_3d="ksat_r_3d"):
    m3d = gj.Map3D(width=800, height=600, use_region=True)
    m3d.render(
        elevation_map="elevation",
        color_map="hydgrp",
        resolution_fine=1,
        perspective=15,
        height=1000,
        zexag=50,
        volume="ksat_r_3d",
        volume_mode="isosurface",
        volume_resolution=3,
        isosurf_level="1:0,1:5,1:20,1:50",
        isosurf_color_map="ksat_r_3d",
        # isosurf_color_value="#3B1F0E,#5A2D1A,#D6A95C,#3A8FB7",
        isosurf_transparency_value="0,30,80,140",
    )
    m3d.overlay.d_legend(raster="ksat_r", title="Ksat (mm/hr)", at=(8, 50, 4, 7))
    m3d.show()

In [ ]:
# | label: ksat_cross_section
# | fig-cap: "Vertical W–E cross-section of Ksat (regular) through the region centre. Depth bins match `depths=0,10,30,60,100` cm above."

reg = tools.g_region(flags="pg", format="json").json
west = float(reg["west"])
east = float(reg["east"])
center_y = (float(reg["north"]) + float(reg["south"])) / 2
ewres = float(reg["ewres"])

profiles = []
for name, title in mapnames_and_title:
    out = tools.r_profile(
        input=name,
        coordinates=f"{west},{center_y},{east},{center_y}",
        resolution=ewres,
        null_value="nan",
    ).text
    vals = []
    for line in out.strip().splitlines():
        parts = line.split()
        if len(parts) < 2:
            continue
        try:
            vals.append(float(parts[1]))
        except ValueError:
            vals.append(np.nan)
    profiles.append(vals)

n = min(len(p) for p in profiles)
arr = np.array([p[:n] for p in profiles])

depth_edges = np.array([0, 10, 30, 60, 100])
dist_edges = np.linspace(0, (east - west) / 1000.0, n + 1)

fig, ax = plt.subplots(figsize=(10, 4))
mesh = ax.pcolormesh(dist_edges, depth_edges, arr, cmap="YlGnBu", shading="flat")
ax.invert_yaxis()
ax.set_xlabel("Distance along transect (km)")
ax.set_ylabel("Depth (cm)")
ax.set_title("Ksat (mm/hr) - W-E cross-section through region centre")
for d in depth_edges:
    ax.axhline(d, color="black", linestyle=":", linewidth=0.8)
fig.colorbar(mesh, ax=ax, label="Ksat (mm/hr)")
plt.show()

## Additional Resources

- [SSURGO Data Access](https://sdmdataaccess.nrcs.usda.gov/)
- [Clay Center, KS – Soil-Informed Overland Flow Simulation](https://ncsu-geoforall-lab.github.io/NRCS_SIMWE/notebooks/ssurgo_clay_center.html)
- [Coweeta, NC – Soil-Informed Overland Flow Simulation](https://ncsu-geoforall-lab.github.io/NRCS_SIMWE/notebooks/ssurgo.html)
